<a href="https://colab.research.google.com/github/fethinho/AI-Trader/blob/main/sarma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# sSARMA BIST SCANNER PRO
# - Multi-thread hızlı tarama
# - Telegram bildirim
# - Sadece YENİ AL verenleri ayrı listeleme
# - UTC+3 (Europe/Istanbul)
# - 15m / 1h / 1d destekli
# - Colab uyumlu
# Pine mantığı korunmuştur:
#   REMA        = WMA(WMA(cprice, length), smooth)
#   REMA_up     = REMA > REMA[1]
#   SwingUp     = REMA_up and not REMA_up[1]
#   SwingDn     = REMA_up[1] and not REMA_up
# ============================================================

!pip -q install yfinance pandas numpy openpyxl requests pytz tabulate

import pandas as pd
import numpy as np
import yfinance as yf
import pytz
import requests
import json
import os
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tabulate import tabulate

# ============================================================
# AYARLAR
# ============================================================

APP_NAME = "sSARMA"
TIMEFRAME = "15m"     # "15m", "1h", "1d"
TZ_NAME = "Europe/Istanbul"

LENGTH = 10
SMOOTH = 2
MULTI = 0.3
SD_LEN = 3

MAX_WORKERS = 16
ONLY_NEW_BUYS = True
SEND_TELEGRAM = True

TELEGRAM_BOT_TOKEN = "8460458441:AAG81QuP0nDc0AT5SYON63bus2d0udab0Iw"
TELEGRAM_CHAT_ID = "314746106"

TELEGRAM_MAX_ROWS = 50
STATE_DIR = os.getcwd()
STATE_FILE = f"{STATE_DIR}/ssarma_state_{TIMEFRAME}.json"

TIMEFRAME_CONFIG = {
    "15m": {"interval": "15m", "period": "60d"},
    "1h":  {"interval": "60m", "period": "730d"},
    "1d":  {"interval": "1d",  "period": "3y"},
}

# ============================================================
# BIST EVRENİ
# ============================================================

BIST_TICKERS_RAW = """
A1CAP.IS, A1YEN.IS, ACSEL.IS, ADEL.IS, ADESE.IS, ADGYO.IS, AEFES.IS, AFYON.IS, AGESA.IS, AGHOL.IS, AGROT.IS,
AHGAZ.IS, AHSGY.IS, AKBNK.IS, AKCNS.IS, AKENR.IS, AKFIS.IS, AKFYE.IS, AKGRT.IS, AKSA.IS, AKSEN.IS, AKSUE.IS,
AKYHO.IS, ALARK.IS, ALBRK.IS, ALCAR.IS, ALCTL.IS, ALFAS.IS, ALKA.IS, ALKIM.IS, ALKLC.IS, ALTIN.IS, ALTNY.IS,
ALVES.IS, ANELE.IS, ANGEN.IS, ANHYT.IS, ANSGR.IS, ARASE.IS, ARCLK.IS, ARDYZ.IS, ARENA.IS, ARDYF.IS, ARMGD.IS,
ARSAN.IS, ARTMS.IS, ARZUM.IS, ASELS.IS, ASTOR.IS, ASUZU.IS, ATAKP.IS, ATATP.IS, ATEKS.IS, ATLAS.IS, ATSYH.IS,
AVGYO.IS, AVHOL.IS, AVOD.IS, AVPGY.IS, AYCES.IS, AYDEM.IS, AYEN.IS, AYES.IS, AYGAZ.IS, AZTEK.IS, BAGFS.IS,
BAHKM.IS, BAKAB.IS, BALAT.IS, BALSU.IS, BANVT.IS, BARMA.IS, BASCM.IS, BASGZ.IS, BAYRK.IS, BEGYO.IS, BESI.IS,
BESLR.IS, BEYAZ.IS, BFREN.IS, BIENY.IS, BIGCH.IS, BIGTK.IS, BIMAS.IS, BINBN.IS, BINHO.IS, BIOEN.IS, BIZIM.IS,
BJKAS.IS, BLCYT.IS, BLUME.IS, BMSCH.IS, BMSTL.IS, BNTAS.IS, BOBET.IS, BORLS.IS, BORSK.IS, BOSSA.IS, BRISA.IS,
BRKO.IS, BRKSN.IS, BRKVY.IS, BRLSM.IS, BRMEN.IS, BRSAN.IS, BRYAT.IS, BSOKE.IS, BTCIM.IS, BUCIM.IS, BULGS.IS,
BURCE.IS, BURVA.IS, BVSAN.IS, BYDNR.IS, CANTE.IS, CASA.IS, CATES.IS, CCOLA.IS, CELHA.IS, CEMAS.IS, CEMTS.IS,
CEMZY.IS, CEOEM.IS, CGCAM.IS, CIMSA.IS, CLEBI.IS, CMBTN.IS, CMENT.IS, CONSE.IS, COSMO.IS, CRDFA.IS, CRFSA.IS,
CUSAN.IS, CVKMD.IS, CWENE.IS, DAGI.IS, DAPGM.IS, DARDL.IS, DCTTR.IS, DENGE.IS, DERHL.IS, DERIM.IS, DESA.IS,
DESPC.IS, DEVA.IS, DGATE.IS, DGNMO.IS, DIRIT.IS, DITAS.IS, DMRGD.IS, DMSAS.IS, DNISI.IS, DOAS.IS, DOFER.IS,
DOFRB.IS, DOGUB.IS, DOHOL.IS, DOKTA.IS, DSTKF.IS, DUNYH.IS, DURDO.IS, DURKN.IS, DYOBY.IS, DZGYO.IS, EBEBK.IS,
ECILC.IS, ECZYT.IS, EDATA.IS, EDIP.IS, EFOR.IS, EGEEN.IS, EGEGY.IS, EGPRO.IS, EGSER.IS, EKIZ.IS, EKOSE.IS,
EKSUN.IS, ELITE.IS, EMKEL.IS, EMNIS.IS, ENKAI.IS, ENSRI.IS, ENTRA.IS, EPLAS.IS, ERBOS.IS, ERCB.IS, EREGL.IS,
ERSU.IS, ESCAR.IS, ESCOM.IS, ESEN.IS, ETILR.IS, ETYAT.IS, EUHOL.IS, EUKYO.IS, EUPWR.IS, EUREN.IS, EUYO.IS,
FADE.IS, FENER.IS, FLAP.IS, FMIZP.IS, FONET.IS, FORM.IS, FORTE.IS, FRIGO.IS, FROTO.IS, FZLGY.IS, GARAN.IS,
GARFA.IS, GEDIK.IS, GEDZA.IS, GENIL.IS, GENTS.IS, GEREL.IS, GESAN.IS, GIPTA.IS, GLBMD.IS, GLCVY.IS, GLRYH.IS,
GLYHO.IS, GMTAS.IS, GOKNR.IS, GOLTS.IS, GOODY.IS, GOZDE.IS, GRNYO.IS, GRSEL.IS, GRTRH.IS, GSDDE.IS, GSDHO.IS,
GSRAY.IS, GUBRF.IS, GUNDG.IS, GWIND.IS, GZNMI.IS, HALKB.IS, HATEK.IS, HATSN.IS, HDFGS.IS, HEDEF.IS, HEKTS.IS,
HKTM.IS, HOROZ.IS, HRKET.IS, HTTBT.IS, HUBVC.IS, HUNER.IS, HURGZ.IS, ICBCT.IS, IEYHO.IS, IHAAS.IS, IHEVA.IS,
IHGZT.IS, IHLAS.IS, IHLGM.IS, IHYAY.IS, IMASM.IS, INDES.IS, INFO.IS, INGRM.IS, INTEK.IS, INTEM.IS, INVEO.IS,
INVES.IS, ISBIR.IS, ISDMR.IS, ISFIN.IS, ISKPL.IS, ISMEN.IS, ISSEN.IS, IZENR.IS, IZFAS.IS, IZINV.IS, IZMDC.IS,
JANTS.IS, KAPLM.IS, KAREL.IS, KARSN.IS, KARTN.IS, KATMR.IS, KAYSE.IS, KBORU.IS, KCAER.IS, KCHOL.IS, KENT.IS,
KERVN.IS, KFEIN.IS, KIMMR.IS, KLKIM.IS, KLMSN.IS, KLNMA.IS, KLRHO.IS, KLSER.IS, KLSYN.IS, KLYPV.IS, KMPUR.IS,
KNFRT.IS, KOCMT.IS, KONKA.IS, KONTR.IS, KONYA.IS, KOPOL.IS, KORDS.IS, KOTON.IS, KRDMD.IS, KRONT.IS, KRPLS.IS,
KRSTL.IS, KRTEK.IS, KRVGD.IS, KSTUR.IS, KTLEV.IS, KTSKR.IS, KUTPO.IS, KUVVA.IS, KZBGY.IS, LIDER.IS, LIDFA.IS,
LILAK.IS, LINK.IS, LKMNH.IS, LMKDC.IS, LOGO.IS, LRSHO.IS, LUKSK.IS, LYDHO.IS, LYDYE.IS, MAALT.IS, MACKO.IS,
MAGEN.IS, MAKIM.IS, MAKTK.IS, MANAS.IS, MARBL.IS, MARKA.IS, MARMR.IS, MARTI.IS, MAVI.IS, MEDTR.IS, MEGAP.IS,
MEGMT.IS, MEKAG.IS, MEPET.IS, MERCN.IS, MERIT.IS, MERKO.IS, METRO.IS, MGROS.IS, MHRGY.IS, MIATK.IS, MMCAS.IS,
MNDRS.IS, MNDTR.IS, MOBTL.IS, MOGAN.IS, MOPAS.IS, MPARK.IS, MRSHL.IS, MTRKS.IS, MTRYO.IS, MZHLD.IS, NATEN.IS,
NETAS.IS, NIBAS.IS, NTGAZ.IS, NTHOL.IS, NUHCM.IS, OBAMS.IS, OBASE.IS, ODAS.IS, ODINE.IS, OFSYM.IS, ONCSM.IS,
ONRYT.IS, ORCAY.IS, ORGE.IS, ORMA.IS, OSMEN.IS, OSTIM.IS, OTKAR.IS, OTTO.IS, OYAKC.IS, OYAYO.IS, OYLUM.IS,
OYYAT.IS, OZATD.IS, OZRDN.IS, OZYSR.IS, PAGYO.IS, PAMEL.IS, PAPIL.IS, PARSN.IS, PASEU.IS, PATEK.IS, PCILT.IS,
PEKGY.IS, PENGD.IS, PENTA.IS, PETKM.IS, PETUN.IS, PGSUS.IS, PINSU.IS, PKART.IS, PKENT.IS, PLTUR.IS, PNLSN.IS,
PNSUT.IS, POLHO.IS, POLTK.IS, PRDGS.IS, PRKAB.IS, PRKME.IS, PRZMA.IS, PSDTC.IS, QNBFK.IS, QNBTR.IS, QUAGR.IS,
RALYH.IS, RAYSG.IS, REEDR.IS, RGYAS.IS, RNPOL.IS, RODRG.IS, RTALB.IS, RUBNS.IS, RUZYE.IS, RYSAS.IS, SAFKR.IS,
SAHOL.IS, SAMAT.IS, SANEL.IS, SANFM.IS, SANKO.IS, SARKY.IS, SASA.IS, SAYAS.IS, SDTTR.IS, SEGYO.IS, SEKFK.IS,
SEKUR.IS, SELVA.IS, SERNT.IS, SEYKM.IS, SILVR.IS, SISE.IS, SKBNK.IS, SKTAS.IS, SKYMD.IS, SKYLP.IS, SMART.IS,
SMRTG.IS, SMRVA.IS, SNICA.IS, SNKRN.IS, SNPAM.IS, SODSN.IS, SOKEY.IS, SOKM.IS, SONME.IS, SRVGY.IS, SUMAS.IS,
SUNTK.IS, SURGY.IS, SUWEN.IS, TABGD.IS, TARKM.IS, TATEN.IS, TATGD.IS, TAVHL.IS, TBORG.IS, TCELL.IS, TCKRC.IS,
TDGYO.IS, TERMI.IS, TEZOL.IS, TGSAS.IS, THYAO.IS, TKFEN.IS, TKNSA.IS, TLMAN.IS, TMPOL.IS, TMSN.IS, TNZTP.IS,
TOASO.IS, TRALT.IS, TRCAS.IS, TRENJ.IS, TRHLO.IS, TRILC.IS, TRMET.IS, TSKB.IS, TSPOR.IS, TTKOM.IS, TTRAK.IS,
TUCLK.IS, TUKAS.IS, TUPRS.IS, TUREX.IS, TURGG.IS, TURSG.IS, UFUK.IS, ULAS.IS, ULKER.IS, ULUFA.IS, ULUSE.IS,
ULUUN.IS, UNLU.IS, USAK.IS, VAKBN.IS, VAKFA.IS, VAKFN.IS, VAKKO.IS, VANGD.IS, VBTYZ.IS, VERUS.IS, VESBE.IS,
VESTL.IS, VKFYO.IS, VKING.IS, VRGYO.IS, VSNMD.IS, YAPRK.IS, YATAS.IS, YAYLA.IS, YBTAS.IS, YEOTK.IS, YESIL.IS,
YGGYO.IS, YIGIT.IS, YKBNK.IS, YKSLN.IS, YONGA.IS, YUNSA.IS, YYAPI.IS, YYLGD.IS, ZEDUR.IS, ZOREN.IS, ZRGYO.IS
"""

def parse_tickers(raw_text):
    parts = [x.strip().upper() for x in raw_text.replace("\n", " ").split(",")]
    cleaned = []
    for p in parts:
        if not p:
            continue
        p = p.replace("İ", "I")
        if ".IS" not in p:
            continue
        if "[" in p or "]" in p:
            continue
        if " " in p:
            p = p.split()[0]
        if p.count(".IS") != 1:
            continue
        if not p.endswith(".IS"):
            continue
        cleaned.append(p)
    return list(dict.fromkeys(cleaned))

BIST_TICKERS = parse_tickers(BIST_TICKERS_RAW)

def get_now_utc3():
    return datetime.now(pytz.timezone(TZ_NAME))

def convert_index_to_utc3(df):
    if df is None or df.empty:
        return df
    tz = pytz.timezone(TZ_NAME)
    if getattr(df.index, "tz", None) is None:
        df.index = df.index.tz_localize("UTC").tz_convert(tz)
    else:
        df.index = df.index.tz_convert(tz)
    return df

def normalize_columns(df):
    if df is None or df.empty:
        return df
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    cols = {}
    for c in df.columns:
        lc = str(c).lower().strip()
        if lc == "open":
            cols[c] = "Open"
        elif lc == "high":
            cols[c] = "High"
        elif lc == "low":
            cols[c] = "Low"
        elif lc == "close":
            cols[c] = "Close"
        elif lc == "adj close":
            cols[c] = "Adj Close"
        elif lc == "volume":
            cols[c] = "Volume"
    return df.rename(columns=cols)

def wma(series, length):
    weights = np.arange(1, length + 1)
    return series.rolling(length).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)

def ssarma_core(df, price_col="Close", length=LENGTH, smooth=SMOOTH, multi=MULTI, sd_len=SD_LEN):
    out = df.copy()
    price = out[price_col].astype(float)
    baseline = wma(price, sd_len)
    dev = multi * price.rolling(sd_len).std()
    upper = baseline + dev
    lower = baseline - dev
    cprice = np.where(price > upper, upper, np.where(price < lower, lower, price))
    cprice = pd.Series(cprice, index=out.index)
    rema = wma(wma(cprice, length), smooth)
    rema_up = rema > rema.shift(1)
    swing_up = rema_up & (~rema_up.shift(1).fillna(False))
    swing_dn = (~rema_up) & (rema_up.shift(1).fillna(False))
    out["sSARMA"] = rema
    out["REMA_UP"] = rema_up
    out["SwingUp"] = swing_up
    out["SwingDn"] = swing_dn
    return out

def fetch_symbol(symbol, interval, period):
    try:
        df = yf.download(
            symbol,
            interval=interval,
            period=period,
            auto_adjust=False,
            progress=False,
            threads=False,
            prepost=False,
        )
        if df is None or df.empty:
            return None
        df = normalize_columns(df)
        df = convert_index_to_utc3(df)
        required = {"Open", "High", "Low", "Close", "Volume"}
        if not required.issubset(df.columns):
            return None
        df = df.dropna(subset=["Open", "High", "Low", "Close"])
        if len(df) < max(30, LENGTH + SMOOTH + SD_LEN + 5):
            return None
        return df
    except Exception:
        return None

def analyze_symbol(symbol, timeframe):
    cfg = TIMEFRAME_CONFIG[timeframe]
    df = fetch_symbol(symbol, cfg["interval"], cfg["period"])
    if df is None or df.empty:
        return None
    sig = ssarma_core(df)
    if len(sig) < 3:
        return None
    last = sig.iloc[-1]
    prev = sig.iloc[-2]
    return {
        "symbol": symbol,
        "ticker": symbol.replace(".IS", ""),
        "timeframe": timeframe,
        "bar_time_utc3": sig.index[-1].strftime("%Y-%m-%d %H:%M:%S %Z%z"),
        "close": round(float(last["Close"]), 4),
        "sSARMA": round(float(last["sSARMA"]), 4) if pd.notna(last["sSARMA"]) else np.nan,
        "rema_up": bool(last["REMA_UP"]) if pd.notna(last["REMA_UP"]) else False,
        "prev_rema_up": bool(prev["REMA_UP"]) if pd.notna(prev["REMA_UP"]) else False,
        "swing_up": bool(last["SwingUp"]) if pd.notna(last["SwingUp"]) else False,
        "swing_dn": bool(last["SwingDn"]) if pd.notna(last["SwingDn"]) else False,
        "signal": "AL" if bool(last["SwingUp"]) else ("SAT" if bool(last["SwingDn"]) else "-"),
        "volume": int(last["Volume"]) if pd.notna(last["Volume"]) else 0,
    }

def scan_market_multithread(symbols, timeframe, max_workers=16):
    rows = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(analyze_symbol, sym, timeframe): sym for sym in symbols}
        total = len(futures)
        done_count = 0
        for future in as_completed(futures):
            done_count += 1
            try:
                row = future.result()
                if row is not None:
                    rows.append(row)
            except Exception:
                pass
            if done_count % 25 == 0 or done_count == total:
                print(f"İşlenen: {done_count}/{total}")
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df = df.sort_values(by=["swing_up", "rema_up", "ticker"], ascending=[False, False, True]).reset_index(drop=True)
    return df

def load_state(path):
    if not os.path.exists(path):
        return {"seen_new_buys": [], "last_run": None}
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return {"seen_new_buys": [], "last_run": None}

def save_state(path, symbols):
    payload = {
        "seen_new_buys": sorted(list(symbols)),
        "last_run": get_now_utc3().strftime("%Y-%m-%d %H:%M:%S %Z%z")
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

def detect_only_new_buy_signals(swing_up_df, state_path):
    state = load_state(state_path)
    prev_seen = set(state.get("seen_new_buys", []))
    current = set(swing_up_df["symbol"].tolist()) if not swing_up_df.empty else set()
    new_only = current - prev_seen
    new_buy_df = swing_up_df[swing_up_df["symbol"].isin(new_only)].copy() if not swing_up_df.empty else pd.DataFrame()
    save_state(state_path, current)
    return new_buy_df, state

def format_pretty_table(df, max_rows=25):
    if df is None or df.empty:
        return "Boş tablo"
    show = df.copy().head(max_rows)
    cols = ["ticker", "signal", "close", "sSARMA", "bar_time_utc3"]
    cols = [c for c in cols if c in show.columns]
    return tabulate(show[cols], headers=cols, tablefmt="github", showindex=False)

def format_telegram_message(timeframe, new_buy_df, all_buy_df):
    now_text = get_now_utc3().strftime("%Y-%m-%d %H:%M")

    if new_buy_df.empty:
        return (
            f"📡 sSARMA TARAMA\n"
            f"⏱ Periyot: {timeframe}\n"
            f"🕒 Saat: {now_text} UTC+3\n"
            f"🟢 Yeni AL: 0 | Aktif AL: {len(all_buy_df)}"
        )

    lines = [
        "📡 sSARMA TARAMA",
        f"⏱ Periyot: {timeframe}",
        f"🕒 Saat: {now_text} UTC+3",
        f"🟢 Yeni AL: {len(new_buy_df)} | Aktif AL: {len(all_buy_df)}",
        ""
    ]

    for _, r in new_buy_df.head(TELEGRAM_MAX_ROWS).iterrows():
        ticker = str(r["ticker"])[:6].ljust(6)
        close_ = f'{float(r["close"]):.2f}'.rjust(7)
        ssarma = f'{float(r["sSARMA"]):.2f}'.rjust(7)
        lines.append(f"🟢 {ticker} {close_} | sS: {ssarma}")

    if len(new_buy_df) > TELEGRAM_MAX_ROWS:
        lines.append("")
        lines.append(f"... +{len(new_buy_df) - TELEGRAM_MAX_ROWS} adet daha")

    return "\n".join(lines)

def send_telegram_message(bot_token, chat_id, text):
    url = f"https://api.telegram.org/bot{bot_token}/sendMessage"
    payload = {"chat_id": chat_id, "text": text, "disable_web_page_preview": True}
    try:
        r = requests.post(url, data=payload, timeout=20)
        return r.status_code, r.text
    except Exception as e:
        return None, str(e)

print(f"{APP_NAME} başlıyor | timeframe={TIMEFRAME} | workers={MAX_WORKERS}")
scan_df = scan_market_multithread(BIST_TICKERS, timeframe=TIMEFRAME, max_workers=MAX_WORKERS)

if scan_df.empty:
    print("Hiç veri dönmedi.")
else:
    swing_up_df = scan_df[scan_df["swing_up"] == True].copy()
    swing_dn_df = scan_df[scan_df["swing_dn"] == True].copy()
    new_buy_df, old_state = detect_only_new_buy_signals(swing_up_df, STATE_FILE)

    print("\n=== ÖZET ===")
    print("Toplam taranan veri satırı:", len(scan_df))
    print("Toplam SwingUp:", len(swing_up_df))
    print("Toplam SwingDn:", len(swing_dn_df))
    print("Sadece YENİ AL:", len(new_buy_df))

    print("\n=== YENİ AL TABLOSU ===")
    display(new_buy_df if ONLY_NEW_BUYS else swing_up_df)

    print("\n=== PRETTY TABLE ===")
    print(format_pretty_table(new_buy_df if ONLY_NEW_BUYS else swing_up_df, max_rows=30))

    ts = get_now_utc3().strftime("%Y%m%d_%H%M%S")
    base = f"{os.getcwd()}/ssarma_{TIMEFRAME}_{ts}"

    scan_df.to_csv(f"{base}_all.csv", index=False, encoding="utf-8-sig")
    swing_up_df.to_csv(f"{base}_swingup.csv", index=False, encoding="utf-8-sig")
    swing_dn_df.to_csv(f"{base}_swingdn.csv", index=False, encoding="utf-8-sig")
    new_buy_df.to_csv(f"{base}_new_buy.csv", index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(f"{base}.xlsx", engine="openpyxl") as writer:
        scan_df.to_excel(writer, sheet_name="ALL_SCAN", index=False)
        swing_up_df.to_excel(writer, sheet_name="SWING_UP", index=False)
        swing_dn_df.to_excel(writer, sheet_name="SWING_DN", index=False)
        new_buy_df.to_excel(writer, sheet_name="NEW_BUY_ONLY", index=False)

    print("\nDosyalar kaydedildi:", base)

    if SEND_TELEGRAM:
        message = format_telegram_message(TIMEFRAME, new_buy_df, swing_up_df)
        status, resp = send_telegram_message(TELEGRAM_BOT_TOKEN, TELEGRAM_CHAT_ID, message)
        print("Telegram durum:", status)
        print("Telegram cevap:", resp[:500] if resp else resp)

sSARMA başlıyor | timeframe=15m | workers=16


/tmp/ipykernel_2255/3751604727.py:187: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  swing_up = rema_up & (~rema_up.shift(1).fillna(False))
/tmp/ipykernel_2255/3751604727.py:188: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  swing_dn = (~rema_up) & (rema_up.shift(1).fillna(False))
/tmp/ipykernel_2255/3751604727.py:187: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_sil

İşlenen: 25/539


/tmp/ipykernel_2255/3751604727.py:187: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  swing_up = rema_up & (~rema_up.shift(1).fillna(False))
/tmp/ipykernel_2255/3751604727.py:188: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  swing_dn = (~rema_up) & (rema_up.shift(1).fillna(False))
/tmp/ipykernel_2255/3751604727.py:187: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_sil

İşlenen: 50/539


...

İşlenen: 525/539


/tmp/ipykernel_2255/3751604727.py:187: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  swing_up = rema_up & (~rema_up.shift(1).fillna(False))
/tmp/ipykernel_2255/3751604727.py:188: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  swing_dn = (~rema_up) & (rema_up.shift(1).fillna(False))
/tmp/ipykernel_2255/3751604727.py:187: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_sil

İşlenen: 539/539

=== ÖZET ===
Toplam taranan veri satırı: 207
Toplam SwingUp: 15
Toplam SwingDn: 13
Sadece YENİ AL: 14

=== YENİ AL TABLOSU ===


,symbol,ticker,timeframe,bar_time_utc3,close,sSARMA,rema_up,prev_rema_up,swing_up,swing_dn,signal,volume
0,AKENR.IS,AKENR,15m,2026-04-15 10:00:00 +03+0300,10.00,9.9800,True,False,True,False,AL,277235
1,BOSSA.IS,BOSSA,15m,2026-04-15 10:00:00 +03+0300,6.45,6.4331,True,False,True,False,AL,66179
2,BRSAN.IS,BRSAN,15m,2026-04-15 10:00:00 +03+0300,580.00,573.6471,True,False,True,False,AL,337318
4,INTEM.IS,INTEM,15m,2026-04-15 10:00:00 +03+0300,283.25,280.3345,True,False,True,False,AL,5125
5,IZFAS.IS,IZFAS,15m,2026-04-15 10:00:00 +03+0300,56.90,55.7397,True,False,True,False,AL,627716
6,KENT.IS,KENT,15m,2026-04-15 09:45:00 +03+0300,415.00,404.0237,True,False,True,False,AL,0
7,MHRGY.IS,MHRGY,15m,2026-04-15 10:00:00 +03+0300,3.64,3.6228,True,False,True,False,AL,392032
8,MIATK.IS,MIATK,15m,2026-04-15 10:00:00 +03+0300,40.46,40.2612,True,False,True,False,AL,1543588
9,MNDTR.IS,MNDTR,15m,2026-04-15 10:00:00 +03+0300,6.06,6.0051,True,False,True,False,AL,71384
10,MOPAS.IS,MOPAS,15m,2026-04-15 10:00:00 +03+0300,42.20,42.0652,True,False,True,False,AL,300970



=== PRETTY TABLE ===
| ticker   | signal   |   close |   sSARMA | bar_time_utc3                |
|----------|----------|---------|----------|------------------------------|
| AKENR    | AL       |   10    |   9.98   | 2026-04-15 10:00:00 +03+0300 |
| BOSSA    | AL       |    6.45 |   6.4331 | 2026-04-15 10:00:00 +03+0300 |
| BRSAN    | AL       |  580    | 573.647  | 2026-04-15 10:00:00 +03+0300 |
| INTEM    | AL       |  283.25 | 280.334  | 2026-04-15 10:00:00 +03+0300 |
| IZFAS    | AL       |   56.9  |  55.7397 | 2026-04-15 10:00:00 +03+0300 |
| KENT     | AL       |  415    | 404.024  | 2026-04-15 09:45:00 +03+0300 |
| MHRGY    | AL       |    3.64 |   3.6228 | 2026-04-15 10:00:00 +03+0300 |
| MIATK    | AL       |   40.46 |  40.2612 | 2026-04-15 10:00:00 +03+0300 |
| MNDTR    | AL       |    6.06 |   6.0051 | 2026-04-15 10:00:00 +03+0300 |
| MOPAS    | AL       |   42.2  |  42.0652 | 2026-04-15 10:00:00 +03+0300 |
| MTRYO    | AL       |    9.78 |   9.6755 | 2026-04-15 10:00:00 +